In [3]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'train_split.parquet')

train_fe_wl = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_train_fe_wl.parquet')

In [6]:
# ============================================================
# 1. 컬럼 분리
# ============================================================

# 시계열 컬럼군
time_cols = [
    'wl_t',          # 수위데이터 대표
    'sfc_rain_1h',   # 육상 누적강수데이터 대표
    'imerg_rain_1h', # 레이더 누적강수데이터 대표
    'radar_reflectivity_mean', # 레이더반사도 대표
    'radar_reflectivity_p95', # 레이더반사도 대표
    'target_maxrise_6h' # 종속변수: 수위상승량
]

# 수위 컬럼군
wl_cols_EDA = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
    'target_maxrise_6h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols_EDA = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h',
    'target_maxrise_6h',
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

print(len(wl_cols_EDA), len(wl_diff_cols_EDA), len(rain_cols), len(env_cols))

7 7 9 7


In [9]:
# ============================================================
# 2. 수위변수 및 종속변수간 상관계수
# ============================================================

print("피어슨 상관계수: 수위변수군")
print(
    train[wl_cols_EDA]
    .corr(
        method='pearson'
    )
    .round(2)
    .to_csv(sep='\t')
)
print()

print("스피어만 상관계수: 수위변수군")
print(
    train[wl_cols_EDA]
    .corr(
        method='spearman'
    )
    .round(2)
    .to_csv(sep='\t')
)
# 정리 필요
# 수위변수군 내부적으로 강력한 상관성이 확인됨
# 그러나 개별 변수의 종속변수에 대한 상관성은 미미함
# -> 개별값보다 시차별 수위 변화에 대한 정보가 예측에 활용됐을 가능성
# -> 차분에 의한 파생변수 생성 검토 필요


피어슨 상관계수: 수위변수군
	wl_t	wl_t_minus_1h	wl_t_minus_3h	wl_t_minus_6h	wl_t_minus_12h	wl_t_minus_24h	target_maxrise_6h
wl_t	1.0	1.0	1.0	1.0	1.0	1.0	-0.03
wl_t_minus_1h	1.0	1.0	1.0	1.0	1.0	1.0	-0.03
wl_t_minus_3h	1.0	1.0	1.0	1.0	1.0	1.0	-0.03
wl_t_minus_6h	1.0	1.0	1.0	1.0	1.0	1.0	-0.02
wl_t_minus_12h	1.0	1.0	1.0	1.0	1.0	1.0	-0.03
wl_t_minus_24h	1.0	1.0	1.0	1.0	1.0	1.0	-0.03
target_maxrise_6h	-0.03	-0.03	-0.03	-0.02	-0.03	-0.03	1.0


스피어만 상관계수: 수위변수군
	wl_t	wl_t_minus_1h	wl_t_minus_3h	wl_t_minus_6h	wl_t_minus_12h	wl_t_minus_24h	target_maxrise_6h
wl_t	1.0	1.0	0.98	0.96	0.96	0.94	-0.01
wl_t_minus_1h	1.0	1.0	0.99	0.96	0.96	0.94	-0.03
wl_t_minus_3h	0.98	0.99	1.0	0.98	0.96	0.94	-0.03
wl_t_minus_6h	0.96	0.96	0.98	1.0	0.96	0.94	-0.01
wl_t_minus_12h	0.96	0.96	0.96	0.96	1.0	0.98	-0.03
wl_t_minus_24h	0.94	0.94	0.94	0.94	0.98	1.0	-0.03
target_maxrise_6h	-0.01	-0.03	-0.03	-0.01	-0.03	-0.03	1.0



In [10]:
# ============================================================
# 3. 수위 차분변수(wl_diff) 및 종속변수 상관계수
# ============================================================

print("피어슨 상관계수: 수위차분(wl_diff)변수군")
print(
    train_fe_wl[wl_diff_cols_EDA]
    .corr(
        method='pearson'
    )
    .round(2)
    .to_csv(sep='\t')
)
print()

print("스피어만 상관계수: 수위차분(wl_diff)변수군")
print(
    train_fe_wl[wl_diff_cols_EDA]
    .corr(
        method='spearman'
    )
    .round(2)
    .to_csv(sep='\t')
)

피어슨 상관계수: 수위차분(wl_diff)변수군
	wl_t	wl_diff_1h	wl_diff_3h	wl_diff_6h	wl_diff_12h	wl_diff_24h	target_maxrise_6h
wl_t	1.0	0.0	0.01	0.01	0.01	0.02	-0.03
wl_diff_1h	0.0	1.0	0.78	0.37	0.18	0.09	0.2
wl_diff_3h	0.01	0.78	1.0	0.73	0.31	0.21	-0.03
wl_diff_6h	0.01	0.37	0.73	1.0	0.49	0.4	-0.29
wl_diff_12h	0.01	0.18	0.31	0.49	1.0	0.8	0.1
wl_diff_24h	0.02	0.09	0.21	0.4	0.8	1.0	0.08
target_maxrise_6h	-0.03	0.2	-0.03	-0.29	0.1	0.08	1.0


스피어만 상관계수: 수위차분(wl_diff)변수군
	wl_t	wl_diff_1h	wl_diff_3h	wl_diff_6h	wl_diff_12h	wl_diff_24h	target_maxrise_6h
wl_t	1.0	0.02	0.05	0.08	0.09	0.1	-0.01
wl_diff_1h	0.02	1.0	0.74	0.54	0.35	0.22	0.4
wl_diff_3h	0.05	0.74	1.0	0.76	0.51	0.35	0.34
wl_diff_6h	0.08	0.54	0.76	1.0	0.68	0.49	0.23
wl_diff_12h	0.09	0.35	0.51	0.68	1.0	0.68	0.2
wl_diff_24h	0.1	0.22	0.35	0.49	0.68	1.0	0.14
target_maxrise_6h	-0.01	0.4	0.34	0.23	0.2	0.14	1.0



In [ ]:
# ============================================================
# 4. 강우변수 및 종속변수 상관계수
# ============================================================

print("피어슨_상관계수: 강우변수군")
print(
    train[rain_cols + ['target_maxrise_6h']]
    .corr(method='pearson')
    .round(2)
    .to_csv(sep='\t')
)
print()

print("스피어만_상관계수: 강우변수군")
print(
    train[rain_cols + ['target_maxrise_6h']]
    .corr(method='spearman')
    .round(2)
    .to_csv(sep='\t')
)

print()

피어슨_상관계수: 강우변수군
	imerg_rain_1h	imerg_rain_6h	imerg_rain_24h	sfc_rain_1h	sfc_rain_6h	sfc_rain_24h	radar_reflectivity_mean	radar_reflectivity_p95	radar_reflectivity_ge20_fraction	target_maxrise_6h
imerg_rain_1h	1.0	0.65	0.4	0.36	0.29	0.2	0.49	0.42	0.47	0.19
imerg_rain_6h	0.65	1.0	0.69	0.42	0.58	0.42	0.5	0.43	0.49	0.2
imerg_rain_24h	0.4	0.69	1.0	0.29	0.5	0.72	0.34	0.31	0.33	0.16
sfc_rain_1h	0.36	0.42	0.29	1.0	0.62	0.39	0.45	0.41	0.43	0.26
sfc_rain_6h	0.29	0.58	0.5	0.62	1.0	0.7	0.38	0.35	0.36	0.25
sfc_rain_24h	0.2	0.42	0.72	0.39	0.7	1.0	0.25	0.26	0.24	0.19
radar_reflectivity_mean	0.49	0.5	0.34	0.45	0.38	0.25	1.0	0.91	0.92	0.19
radar_reflectivity_p95	0.42	0.43	0.31	0.41	0.35	0.26	0.91	1.0	0.82	0.18
radar_reflectivity_ge20_fraction	0.47	0.49	0.33	0.43	0.36	0.24	0.92	0.82	1.0	0.18
target_maxrise_6h	0.19	0.2	0.16	0.26	0.25	0.19	0.19	0.18	0.18	1.0


스피어만_상관계수: 강우변수군
	imerg_rain_1h	imerg_rain_6h	imerg_rain_24h	sfc_rain_1h	sfc_rain_6h	sfc_rain_24h	radar_reflectivity_mean	radar_reflectivity_p95	ra

In [ ]:
# ============================================================
# 5. 유역환경변수 및 종속변수간 상관계수
# ============================================================

print("피어슨_상관계수: 환경변수군")
print(
    train[env_cols + ['target_maxrise_6h']]
    .corr(method='pearson')
    .round(2)
    .to_csv(sep='\t')
)
print()

print("스피어만_상관계수: 환경변수군")
print(
    train[env_cols + ['target_maxrise_6h']]
    .corr(method='spearman')
    .round(2)
    .to_csv(sep='\t')
)

print()

피어슨_상관계수: 환경변수군
	cat_area	wsarea	mean_elevation	mean_slope	urban_ratio	forest_ratio	stream_order	target_maxrise_6h
cat_area	1.0	0.58	-0.17	-0.2	0.5	-0.27	0.45	0.03
wsarea	0.58	1.0	-0.19	-0.14	0.17	-0.18	0.74	0.02
mean_elevation	-0.17	-0.19	1.0	0.79	-0.39	0.7	-0.2	-0.04
mean_slope	-0.2	-0.14	0.79	1.0	-0.49	0.94	-0.15	-0.05
urban_ratio	0.5	0.17	-0.39	-0.49	1.0	-0.59	0.06	0.03
forest_ratio	-0.27	-0.18	0.7	0.94	-0.59	1.0	-0.16	-0.06
stream_order	0.45	0.74	-0.2	-0.15	0.06	-0.16	1.0	0.01
target_maxrise_6h	0.03	0.02	-0.04	-0.05	0.03	-0.06	0.01	1.0


스피어만_상관계수: 환경변수군
	cat_area	wsarea	mean_elevation	mean_slope	urban_ratio	forest_ratio	stream_order	target_maxrise_6h
cat_area	1.0	0.21	-0.32	-0.3	0.37	-0.33	0.26	0.04
wsarea	0.21	1.0	-0.26	-0.18	-0.11	-0.17	0.89	0.05
mean_elevation	-0.32	-0.26	1.0	0.88	-0.58	0.85	-0.23	-0.1
mean_slope	-0.3	-0.18	0.88	1.0	-0.65	0.96	-0.19	-0.07
urban_ratio	0.37	-0.11	-0.58	-0.65	1.0	-0.72	-0.05	0.04
forest_ratio	-0.33	-0.17	0.85	0.96	-0.72	1.0	-0.18	-0.08
stream_ord